# OpenPlaque V23 — CT-only heart/aorta anatomical-restriction experiment

**Goal:** Test whether a heart/aorta region derived independently by TotalSegmentator reduces the implausibly large whole-volume V22 predictions while retaining the existing coronary positive annotations. Uses the frozen **V21 model predictions** from **V22**. **No coronary model training.**

**Critical data fact:** the older TotalSegmentator caches belong to the *original validation 80*, **not** the V21/V22 internal DEV80. This notebook creates new patient-specific masks for DEV80 and never copies masks between cases.

A fixed 12-case pilot runs first. Predeclared gate at 20 mm: retain at least **98%** of existing V22 annotated-positive hits **for every artery**, and remove at least **50%** of raw predicted head-voxels overall. If the gate fails, the experiment stops after the pilot. If it passes, the same run continues through all 80. Reruns reuse validated masks and per-case reports. The 10/20/30-mm radii are descriptive; no anatomy is auto-promoted.

**Safety and scope:** The region is made only from a CT-derived whole-heart mask and an aorta mask (TotalSegmentator total / fast). No reference-derived crop, no original training or test images, no mask rewrites, no use of unlabeled tissue as known negatives. Passing this gate is **not** a clinical or coronary identity validation. Earlier V12/V13 heart ROI screens remained anatomically nonspecific.

**Run:** T4 or L4 GPU → Runtime → Run all. The first executable cell mounts Google Drive; reuse/configuration immediately follows. TotalSegmentator weights may be downloaded on first use; interrupted cases safely resume from Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cache / run controls. Do not change the frozen split, predictions, or pilot gate.
REUSE_VERIFIED_MASKS = True
RUN_FULL_80_IF_PILOT_PASSES = True
RUN_UNIT_TESTS = True
RUN_PILOT_AND_CONDITIONAL_FULL = True
print('V23: frozen DEV80; 12-case predefined pilot; CT-only anatomical ROIs.')


In [ ]:
import sys, subprocess, importlib.util, importlib.metadata
for package in ('nibabel','scipy'):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
try:
    ts_version=importlib.metadata.version('TotalSegmentator')
except importlib.metadata.PackageNotFoundError:
    ts_version=None
if ts_version!='2.11.0':
    print('Installing TotalSegmentator 2.11.0 for consistent anatomy generation')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','TotalSegmentator==2.11.0'])
import torch, nibabel, scipy
assert torch.cuda.is_available(),'STOP: T4/L4 GPU required for TotalSegmentator heart/aorta task'
print('CUDA:',torch.cuda.get_device_name(0),'TotalSegmentator:',importlib.metadata.version('TotalSegmentator'))


In [ ]:
%%writefile /content/openplaque_v23_anatomic_roi.py
"""OpenPlaque V23: CT-only anatomical region counterfactual for V22 frozen DEV80.

Research diagnostic only. TotalSegmentator masks are model estimates, not validated anatomy.
No source/reference edits, manual geometry, held-out test access, retraining, or label-derived ROIs.
"""
import argparse
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path

VERSION = 'openplaque-v23-imagecas-dev80-heart-aorta-candidate-screen-20261013-1'
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
BASE = ROOT/'OpenPlaque_Evaluation_Dataset_v0_2'
OLD = BASE/'evaluation_v22_full_volume'
OUT = BASE/'evaluation_v23_heart_aorta_roi'
CT_ROOT = ROOT/'ExternalData/ImageCAS_OriginalCCTA_MatchedTo_ImageCASX_v1/volumes'
SEGS = ROOT/'Cache/ImageCAS_X_Pretrained_v1/ImageCAS-X_dataset.zip_extracted/ImageCAS-X_dataset/segmentations'
DEV_IDS = BASE/'internal_dev_80_frozen.txt'
FROZEN_DEV_SHA = 'ef2bd3bdbe7dc9cbf4639e62d80dceb53108ac836acff64617e0f17b4f29be4f'
V21_CHECKPOINT_SHA = 'f789eede621816433240232455d7a7862c7eafc69a979a50938e73349a6796b9'
HEADS = {'LM': (1, 0), 'LAD': (2, 1), 'LCX': (3, 2), 'RCA': (9, 3)}
RADII_MM = (10.,20.,30.)
PILOT_IDS = ('11','142','203','305','370','455','562','633','708','776','853','976')
GATE_RADIUS_MM = 20.
MIN_PER_ARTERY_RETENTION = .98   # of V22 true positives, for each of four classes
MIN_WEIGHTED_PRED_REMOVAL = .50 # of raw four-head positive voxel count
MAX_ANALYSIS_RADIUS_MM = 42.


def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()


def save_json(p,obj):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    q=p.with_name(p.name+'.tmp')
    q.write_text(json.dumps(obj,indent=2,sort_keys=True,allow_nan=False),encoding='utf8')
    os.replace(q,p)


def load_json(p):return json.loads(Path(p).read_text())


def run_preflight():
    ids=[s.strip() for s in DEV_IDS.read_text().splitlines() if s.strip()]
    assert len(ids)==len(set(ids))==80 and all(i.isdigit() for i in ids),'Corrupt DEV80 case list'
    canonical=''.join(x+'\n' for x in sorted(ids,key=int)).encode()
    if hashlib.sha256(canonical).hexdigest()!=FROZEN_DEV_SHA:
        raise RuntimeError('STOP: frozen DEV80 membership hash mismatch')
    if not set(PILOT_IDS)<=set(ids):raise RuntimeError('PILOT includes non DEV80 case')
    v22=load_json(OLD/'whole_volume_summary_v22.json')
    m22=load_json(OLD/'manifest_v22.json')
    if (not v22.get('complete') or v22.get('cases_complete')!=80
        or v22.get('missing_ids') or v22.get('frozen_dev_sha256')!=FROZEN_DEV_SHA
        or v22.get('model_checkpoint_sha256')!=V21_CHECKPOINT_SHA):
        raise RuntimeError('STOP: V22 DEV80 completion/provenance mismatch')
    if m22.get('dev_ids_sha256')!=FROZEN_DEV_SHA or m22.get('model_checkpoint_sha256')!=V21_CHECKPOINT_SHA:
        raise RuntimeError('STOP: V22 mask model/split mismatch')
    q=load_json(BASE/'sparse_topology_results_v0_3/frozen_positive_reference_manifest_v0_3.json')
    if q.get('status')!='FROZEN_POSITIVE_REFERENCE_QA_ONLY' or q.get('sealed_test_accessed') is not False:
        raise RuntimeError('STOP: V0.3 frozen annotation provenance missing')
    cfg={
        'version':VERSION,'case_ids_sha256':FROZEN_DEV_SHA,'model_checkpoint_sha256':V21_CHECKPOINT_SHA,
        'source_v22':'evaluation_v22_full_volume','total_task':'total','total_fast':True,
        'total_roi_subset':['heart','aorta'],'total_version_target':'2.11.0',
        'radii_mm':list(RADII_MM),'aorta_root_support':'distance_to_aorta_le_8mm AND distance_to_heart_le_r_plus_12mm',
        'pilot_ids':list(PILOT_IDS),'pilot_gate_radius_mm':GATE_RADIUS_MM,
        'pilot_gate_min_retention_vs_raw_per_artery':MIN_PER_ARTERY_RETENTION,
        'pilot_gate_min_aggregate_raw_prediction_volume_removed':MIN_WEIGHTED_PRED_REMOVAL,
        'inference_region_not_derived_from_reference':True,
        'reference_masks_only_for_post_hoc_metrics':True,
        'aorta_or_heart_prediction_is_not_ground_truth':True,
        'new_model_training':False,'test_reads':0,'source_writes':0,'label_zero_not_a_verified_negative':True,
        'comparison_not_independent_clinical_validation':True
    }
    outmanifest=OUT/'manifest_v23.json';OUT.mkdir(parents=True,exist_ok=True)
    if outmanifest.exists():
        if load_json(outmanifest)!=cfg:raise RuntimeError('STOP: V23 manifest differs. Do not mix runs.')
    else:save_json(outmanifest,cfg)
    for cid in ids:
        if not (OLD/'per_case'/f'{cid}.json').exists():raise RuntimeError('V22 per-case missing: '+cid)
        if not (OLD/'prediction_masks'/f'{cid}.v22_4head_native_raw.nii.gz').exists():
            raise RuntimeError('V22 raw mask missing: '+cid)
    return ids,cfg


def ensure_anatomy(cid,ctpath):
    """Generate patient-specific heart/aorta masks, cached and SHA verified.
    Uses only CT; annotation labels must NEVER enter this function.
    """
    p=OUT/'totalseg_masks'/cid
    valid=p/'verified_task.json'
    ct_fp=sha(ctpath)
    if valid.exists():
        m=load_json(valid)
        if (m.get('ct_sha256')==ct_fp and m.get('version')==VERSION and
            all((p/f'{name}.nii.gz').is_file() and sha(p/f'{name}.nii.gz')==m['masks_sha256'].get(name) for name in ('heart','aorta'))):
            return p,'REUSED'
    cmd=shutil.which('TotalSegmentator') or shutil.which('totalseg')
    if not cmd:raise RuntimeError('TotalSegmentator CLI unavailable; install TotalSegmentator==2.11.0')
    p.mkdir(parents=True,exist_ok=True)
    tmp=p/'_incomplete_tmp'
    if tmp.exists():shutil.rmtree(tmp)
    cmdline=[cmd,'-i',str(ctpath),'-o',str(tmp),'-ta','total','--fast',
             '--roi_subset','heart','aorta','--device','gpu','--nr_thr_saving','1']
    with (p/'inference.log').open('w') as f:
        r=subprocess.run(cmdline,stdout=f,stderr=subprocess.STDOUT)
    if r.returncode:
        raise RuntimeError(f'TotalSegmentator failed case {cid} ({r.returncode}); read {p}/inference.log')
    for name in ('heart','aorta'):
        q=tmp/f'{name}.nii.gz'
        if not q.is_file() or q.stat().st_size<100:raise RuntimeError(f'TotalSegmentator missing {name} case {cid}')
    for name in ('heart','aorta'):
        os.replace(tmp/f'{name}.nii.gz',p/f'{name}.nii.gz')
    shutil.rmtree(tmp)
    m={'case':cid,'version':VERSION,'ct_sha256':ct_fp,'masks_sha256':{
       k:sha(p/f'{k}.nii.gz') for k in ('heart','aorta')},'source_mask_reference_used':False,
       'task':'total','fast':True,'subset':['heart','aorta'],'status':'COMPLETE'}
    save_json(valid,m)
    return p,'NEW'


def crop_around(mask,spacing,padding_mm=MAX_ANALYSIS_RADIUS_MM):
    import numpy as np
    pts=np.where(mask)
    if len(pts[0])==0:raise RuntimeError('Whole-heart segmentation empty')
    pad=np.ceil(padding_mm/np.asarray(spacing)).astype(int)+3
    lo=np.maximum(0,np.array([int(v.min()) for v in pts])-pad)
    hi=np.minimum(mask.shape,np.array([int(v.max())+1 for v in pts])+pad)
    return tuple(slice(int(a),int(b)) for a,b in zip(lo,hi))


def roi_distance_masks(heart_crop,aorta_crop,spacing):
    from scipy.ndimage import distance_transform_edt
    if not heart_crop.any():raise RuntimeError('Empty crop heart')
    dist_h=distance_transform_edt(~heart_crop,sampling=spacing).astype('float32')
    if not aorta_crop.any():raise RuntimeError('Aorta missing from ROI crop')
    dist_a=distance_transform_edt(~aorta_crop,sampling=spacing).astype('float32')
    return dist_h,dist_a


def anatomical_roi(dist_h,dist_a,radius):
    """No reference labels or model predictions used to define the anatomical region."""
    return (dist_h<=radius)|((dist_a<=8.)&(dist_h<=radius+12.))


def native_mask(path,target):
    import nibabel as nib
    import numpy as np
    from nibabel.processing import resample_from_to
    m=nib.load(str(path))
    if m.shape!=target.shape or not np.allclose(m.affine,target.affine,atol=1e-3):
        m=resample_from_to(m,(target.shape,target.affine),order=0)
    a=np.asarray(m.dataobj,dtype=np.uint8)>0
    if a.shape!=target.shape:raise RuntimeError('Anatomy geometry resample failed')
    return a


def case_report(cid):
    import numpy as np
    import nibabel as nib
    from scipy import ndimage as ndi
    cp=OUT/'per_case'/f'{cid}.json'
    old=load_json(OLD/'per_case'/f'{cid}.json')
    ctpath=CT_ROOT/f'{cid}.img.nii.gz'
    refpath=SEGS/f'{cid}.coronary.nii.gz'
    predpath=OLD/'prediction_masks'/f'{cid}.v22_4head_native_raw.nii.gz'
    if not all(p.is_file() for p in (ctpath,refpath,predpath)):raise RuntimeError('Source case missing: '+cid)
    predsha=sha(predpath)
    maskroot,state=ensure_anatomy(cid,ctpath)
    anatomysha={k:sha(maskroot/f'{k}.nii.gz') for k in ('heart','aorta')}
    if cp.exists():
        oldreport=load_json(cp)
        if (oldreport.get('version')==VERSION and oldreport.get('v22_mask_sha256')==predsha
            and oldreport.get('anatomy_sha256')==anatomysha):return oldreport,'REUSED'
    ctnii=nib.load(str(ctpath)); refnii=nib.load(str(refpath)); prednii=nib.load(str(predpath))
    if (ctnii.shape!=refnii.shape or ctnii.shape!=prednii.shape or
        not np.allclose(ctnii.affine,refnii.affine,atol=1e-3) or
        not np.allclose(ctnii.affine,prednii.affine,atol=1e-3)):
        raise RuntimeError('Source CT, annotations and V22 mask differ in shape/affine: '+cid)
    bits=np.asarray(prednii.dataobj,dtype=np.uint8)
    if np.any(bits>15):raise RuntimeError('Corrupted four-head bitmask')
    spacing=tuple(map(float,ctnii.header.get_zooms()[:3]));voxel_mm3=float(np.prod(spacing))
    heart=native_mask(maskroot/'heart.nii.gz',ctnii)
    aorta=native_mask(maskroot/'aorta.nii.gz',ctnii)
    crop=crop_around(heart,spacing)
    heart_c=heart[crop];aorta_c=aorta[crop]
    dh,da=roi_distance_masks(heart_c,aorta_c,spacing)
    del heart,aorta,heart_c,aorta_c
    seg=np.asarray(refnii.dataobj,dtype=np.uint8)[crop]
    hu=np.asarray(ctnii.dataobj,dtype=np.float32)[crop]
    bits_crop=bits[crop]
    if not np.isfinite(hu).all():raise RuntimeError('Nonfinite source CT')
    per_radius={};eligible={name:bool(old['per_artery'][name]['reference_positive_eligible']) for name in HEADS}
    for radius in RADII_MM:
        roi=anatomical_roi(dh,da,radius)
        metrics={};n_ret=0;d_ret=0;n_pred=0;d_pred=0
        for name,(lab,j) in HEADS.items():
            raw=old['per_artery'][name];pred=((bits_crop&(1<<j))!=0)&roi
            pred_n=int(pred.sum());pos=(seg==lab)
            hit=int((pred&pos).sum());raw_hit=int(raw['annotated_positive_hits']) if raw['annotated_positive_hits'] is not None else None
            low_hu=int((pred&(hu< -300)).sum())
            raw_n=int(raw['raw_prediction_voxels'])
            # Six-neighbor 3D connectivity in CT-only crop; never uses masks for topology.
            structure=ndi.generate_binary_structure(3,1)
            cc,ncc=ndi.label(pred,structure=structure)
            counts=np.bincount(cc.ravel(),minlength=1)[1:]
            major=int(np.count_nonzero(counts*voxel_mm3>=10.))
            largest=int(counts.max()) if len(counts) else 0
            del cc,counts
            known_other=np.zeros(seg.shape,dtype=bool)
            if eligible[name]:
                for other,(olab,_) in HEADS.items():
                    if other!=name and eligible[other]:known_other|=(seg==olab)
            other_hit=int((pred&known_other).sum()) if eligible[name] else None
            known_pos=int(raw['reference_positive_voxels']) if raw['reference_positive_voxels'] is not None else None
            retain=(hit/raw_hit if raw_hit and eligible[name] else None)
            if eligible[name] and raw_hit:
                n_ret+=hit;d_ret+=raw_hit
            n_pred+=pred_n;d_pred+=raw_n
            metrics[name]={
                'prediction_voxels':pred_n,'prediction_mm3':pred_n*voxel_mm3,
                'raw_prediction_voxels':raw_n,'fraction_of_raw_predictions_removed':1-pred_n/max(1,raw_n),
                'reference_class_eligible':eligible[name],'reference_positive_n':known_pos,
                'hit_on_known_positive':hit if eligible[name] else None,
                'recall_of_reference_positive':hit/known_pos if eligible[name] and known_pos else None,
                'fraction_of_raw_true_positive_hits_retained':retain,
                'prediction_below_minus300HU_voxels':low_hu,
                'prediction_below_minus300HU_fraction':low_hu/max(1,pred_n),
                'predictions_on_known_other_major_voxels':other_hit,
                'predicted_components_6connected':int(ncc),
                'predicted_components_ge_10mm3':major,
                'largest_predicted_component_fraction':largest/max(1,pred_n),
                'unannotated_not_verified_negative':True}
        per_radius[str(int(radius))]={'per_artery':metrics,
              'raw_predicted_head_voxels':d_pred,
              'retained_predicted_head_voxels':n_pred,
              'weighted_predicted_volume_reduction':1-n_pred/max(1,d_pred),
              'eligible_known_positive_hit_retention':n_ret/max(1,d_ret),
              'unprotected_reference_fraction_outside_roi':{
                 n:float(((seg==lab)&(~roi)).sum()/max(1,(seg==lab).sum())) for n,(lab,_) in HEADS.items()}}
        del roi
    out={'version':VERSION,'case_id':cid,'v22_checkpoint_sha256':V21_CHECKPOINT_SHA,
         'v22_mask_sha256':predsha,'anatomy_sha256':anatomysha,
         'native_spacing_mm':list(spacing),'native_shape':list(ctnii.shape),
         'anatomy_mask_state':state,'crop_shape':[v.stop-v.start for v in crop],
         'radii_mm':per_radius,'new_training':False,'test_reads':0,'source_modified':False,
         'reference_used_for_region_generation':False,'verified_negative_voxels':0,
         'model_masks_are_uncertain':True}
    save_json(cp,out)
    return out,'NEW'


def aggregate(ids):
    import numpy as np
    found=[];miss=[]
    for cid in ids:
        p=OUT/'per_case'/f'{cid}.json'
        if not p.is_file():miss.append(cid);continue
        x=load_json(p)
        if x.get('version')!=VERSION:miss.append(cid);continue
        found.append(x)
    summaries={}
    for r in RADII_MM:
        key=str(int(r)); vals=[v['radii_mm'][key] for v in found]
        d={'n':len(vals),'weighted_prediction_removed':sum(v['raw_predicted_head_voxels']-v['retained_predicted_head_voxels'] for v in vals)/max(1,sum(v['raw_predicted_head_voxels'] for v in vals)),
           'reference_eligible_hit_retention':sum(v['eligible_known_positive_hit_retention']*sum(
                (v['per_artery'][name]['raw_prediction_voxels']*0+1) for name in HEADS) for v in []) if False else None}
        class_summary={}
        for name in HEADS:
            m=[v['per_artery'][name] for v in vals]
            eligible=[x for x in m if x['reference_class_eligible'] and x['fraction_of_raw_true_positive_hits_retained'] is not None]
            # Number of hits in original V22 for a case is hit_retained/ratio;
            # aggregate exactly by reading V22 per-case reports, not by inferring from rounded values.
            ids_present=[v['case_id'] for v in found if v['radii_mm'][key]['per_artery'][name]['reference_class_eligible']]
            n_hit=sum(v['radii_mm'][key]['per_artery'][name]['hit_on_known_positive'] or 0 for v in found)
            raw_hit=sum(load_json(OLD/'per_case'/f'{cid}.json')['per_artery'][name]['annotated_positive_hits'] or 0 for cid in ids_present)
            class_summary[name]={
                'n_eligible':len(ids_present),
                'raw_true_positive_hits':raw_hit,'retained_true_positive_hits':n_hit,
                'fraction_raw_true_positive_hits_retained':n_hit/max(1,raw_hit),
                'raw_prediction_voxels':sum(x['raw_prediction_voxels'] for x in m),
                'retained_prediction_voxels':sum(x['prediction_voxels'] for x in m),
                'fraction_raw_prediction_removed':1-sum(x['prediction_voxels'] for x in m)/max(1,sum(x['raw_prediction_voxels'] for x in m)),
                'median_components_ge_10mm3':float(np.median([x['predicted_components_ge_10mm3'] for x in m])) if m else None,
                'median_unprotected_reference_fraction_outside_roi':float(np.median([v['unprotected_reference_fraction_outside_roi'][name] for v in vals])) if vals else None}
        d['per_artery']=class_summary
        d['reference_eligible_hit_retention']=sum(v['retained_true_positive_hits'] for v in class_summary.values())/max(1,sum(v['raw_true_positive_hits'] for v in class_summary.values()))
        summaries[key]=d
    pilot_present=set(PILOT_IDS)<=set(x['case_id'] for x in found)
    # Pilot gate uses EXACTLY the predefined 12 cases, not the whole available selection.
    gate=None
    if pilot_present:
        pilot=aggregate_pilot(found)
        gate=pilot['gate']
    return {'version':VERSION,'n_complete':len(found),'n_expected':len(ids),'missing_case_ids':miss,
       'radii_summary':summaries,'predeclared_pilot':gate,'test_reads':0,'no_new_training':True,
       'unannotated_predictions_remain_unknown':True,
       'roi_inference_uses_only_ct_derived_totalseg_masks':True,
       'previous_v12_v13_experiments_in_original_validation_not_independent':True}


def aggregate_pilot(found):
    pilot=[v for v in found if v['case_id'] in PILOT_IDS]
    if len(pilot)!=len(PILOT_IDS):return {'complete':False}
    radius=str(int(GATE_RADIUS_MM))
    per_class={};raw=0;remaining=0
    for x in pilot:
        a=x['radii_mm'][radius]
        raw+=a['raw_predicted_head_voxels'];remaining+=a['retained_predicted_head_voxels']
    for name in HEADS:
        hits=0;raw_hits=0
        for x in pilot:
            e=x['radii_mm'][radius]['per_artery'][name]
            if e['reference_class_eligible']:
                hits+=e['hit_on_known_positive'] or 0
                raw_hits+=load_json(OLD/'per_case'/f"{x['case_id']}.json")['per_artery'][name]['annotated_positive_hits'] or 0
        per_class[name]=hits/max(1,raw_hits)
    removal=1-remaining/max(1,raw)
    passed=all(v>=MIN_PER_ARTERY_RETENTION for v in per_class.values()) and removal>=MIN_WEIGHTED_PRED_REMOVAL
    return {'gate':{'complete':True,'passed':passed,'radius_mm':GATE_RADIUS_MM,
            'per_class_true_positive_retention':per_class,
            'weighted_predicted_voxel_reduction':removal,
            'min_retention_requirement':MIN_PER_ARTERY_RETENTION,
            'min_prediction_reduction_requirement':MIN_WEIGHTED_PRED_REMOVAL,
            'next_step':'PROCESS_ALL_80' if passed else 'HALT_FULL_80_AND_REPORT',
            'anatomical_identity_established':False}}


def save_summary(ids,stage='pilot'):
    result=aggregate(ids)
    result['stage']=stage;save_json(OUT/'summary_v23.json',result)
    page=['<!doctype html><meta charset="utf-8"><title>OpenPlaque V23 Anatomy Restriction</title>',
        '<style>body{font:16px/1.55 system-ui;max-width:950px;margin:30px auto;padding:15px}pre{white-space:pre-wrap;background:#eee;padding:18px}</style>',
        '<h1>V23 — ImageCAS-X Heart + Aorta ROI screen</h1>',
        '<p>Research diagnostic only. Source CT and annotations unchanged; reference labels are incomplete; never report unlabeled predictions as proven false positives.</p>',
        f'<h2>Completion: {result["n_complete"]}/{result["n_expected"]}</h2>',
        '<p>A predeclared pilot gate controls whether the remaining scans are processed. A large reduction in prediction volume is not proof of correct artery identification.</p>',
        '<pre>'+json.dumps(result,indent=2)+'</pre>']
    html=OUT/'OPENPLAQUE_V23_ANATOMY_RESTRICTION_REPORT.html'
    tmp=html.with_name(html.name+'.tmp');tmp.write_text('\n'.join(page));os.replace(tmp,html)
    return result


def run(full_if_gate):
    ids,_=run_preflight()
    import torch
    if not torch.cuda.is_available():raise RuntimeError('GPU required to generate patient-specific anatomy masks')
    errors=[]
    for i,cid in enumerate(PILOT_IDS,1):
        try:
            r,state=case_report(cid);print(f'PILOT {i}/12 {cid}: {state}',flush=True)
        except Exception as e:
            errors.append({'case':cid,'error':str(e),'traceback':traceback.format_exc()[-1200:]})
            save_json(OUT/'failures_v23.json',errors)
            raise
    result=save_summary(ids,stage='pilot')
    gate=result.get('predeclared_pilot')
    print('PILOT GATE',json.dumps(gate,indent=2),flush=True)
    if not gate or not gate.get('passed'):
        print('Predeclared V23 gate FAILED: full 80 intentionally skipped. No model promotion.',flush=True)
        save_json(OUT/'run_status_v23.json',{'status':'PILOT_GATE_FAILED_FULL_NOT_RUN','gate':gate})
        return
    if not full_if_gate:
        print('Pilot completed; full run disabled in config.',flush=True)
        return
    for i,cid in enumerate(ids,1):
        if cid in PILOT_IDS:continue
        try:
            _,state=case_report(cid);print(f'FULL {i}/{len(ids)} {cid}: {state}',flush=True)
            if i%5==0:save_summary(ids,stage='full_in_progress')
        except Exception as e:
            errors.append({'case':cid,'error':str(e),'traceback':traceback.format_exc()[-1200:]})
            save_json(OUT/'failures_v23.json',errors)
            raise
    result=save_summary(ids,stage='full_complete')
    save_json(OUT/'run_status_v23.json',{'status':'COMPLETE_80_RESEARCH_DIAGNOSTIC','n_complete':result['n_complete'],'gate':gate})
    print('V23 COMPLETE',result['n_complete'],'cases',flush=True)


def selftest():
    import numpy as np
    from scipy.ndimage import distance_transform_edt
    mask=np.zeros((26,28,20),bool);mask[8:15,10:18,5:12]=1
    aorta=np.zeros_like(mask);aorta[12:14,14:16,4:9]=True
    sp=(.5,.5,1.)
    crop=crop_around(mask,sp,20)
    dh,da=roi_distance_masks(mask[crop],aorta[crop],sp)
    for r in RADII_MM:
        roi=anatomical_roi(dh,da,r)
        assert roi[mask[crop]].all(), 'Heart interior omitted'
    assert anatomical_roi(dh,da,10).sum()<=anatomical_roi(dh,da,20).sum()<=anatomical_roi(dh,da,30).sum()
    assert all(k in HEADS for k in ('LM','LAD','LCX','RCA'))
    # Connected component sanity: two separated blobs must remain two.
    from scipy.ndimage import label,generate_binary_structure
    test=np.zeros((15,15,15),bool);test[1:3,1:3,1:3]=1;test[12:14,12:14,12:14]=1
    assert label(test,structure=generate_binary_structure(3,1))[1]==2
    print('PASS: CT-only ROI nesting, boundary-safe bbox, physical distance, 6-connected components')


def main():
    pa=argparse.ArgumentParser()
    pa.add_argument('stage',choices=('selftest','preflight','run','report'))
    pa.add_argument('--run-full-after-pilot',action='store_true')
    args=pa.parse_args()
    if args.stage=='selftest':selftest();return
    if args.stage=='preflight':
        ids,manifest=run_preflight();print('PASS source manifest',len(ids),manifest['version']);return
    if args.stage=='run':run(args.run_full_after_pilot);return
    if args.stage=='report':
        ids,_=run_preflight();r=save_summary(ids,stage='report');print('Status',r['n_complete'],'/',r['n_expected']);return

if __name__=='__main__':main()


In [ ]:
from pathlib import Path
import subprocess, sys, hashlib
V23 = Path('/content/openplaque_v23_anatomic_roi.py')
EXPECTED_SHA='d40eef6e4f8d2ea8f7f3919281ebd61bae9fef528328ab3b2ad39662f7355ddf'
assert hashlib.sha256(V23.read_bytes()).hexdigest()==EXPECTED_SHA,'STOP: V23 code fingerprint changed'
def stage(*args):
    subprocess.run([sys.executable,str(V23),*list(map(str,args))],check=True)
if RUN_UNIT_TESTS:stage('selftest')
stage('preflight')
print('Pinned evaluator:',EXPECTED_SHA)


In [ ]:
if RUN_PILOT_AND_CONDITIONAL_FULL:
    args=['run']
    if RUN_FULL_80_IF_PILOT_PASSES:args+=['--run-full-after-pilot']
    stage(*args)


In [ ]:
# Summarize partial or completed result even when full 80 was gated off.
from pathlib import Path
import json
p=Path('/content/drive/MyDrive/OpenPlaque/OpenPlaque_Evaluation_Dataset_v0_2/evaluation_v23_heart_aorta_roi')
print('Outputs:',p)
for name in ('run_status_v23.json','summary_v23.json','manifest_v23.json'):
    f=p/name
    if f.is_file():print(name, json.dumps(json.loads(f.read_text()),indent=2)[:12000])
print('V23 is a research counterfactual. Do not promote a coronary identity from ROI containment.')
